## **Perceptron Multicamadas (MLP)**

Uma rede feedforward com $L$ camadas computa uma função $y_k(\mathbf{x}, \mathbf{w})$ onde $\mathbf{w}$ é o vetor de todos os parâmetros.

A ativação de cada unidade na camada $\ell$ é dada por:

$$
a_j^{(\ell)} = \sum_i w_{ji}^{(\ell)} \cdot z_{i}^{(\ell-1)}
\quad
\text{ e }
\quad
z_{j}^{(\ell)} = h\left(a_j^{(\ell)}\right)
$$

onde:

* A função $h(\cdot)$ é a função de ativação ($\tanh$ ou sigmoide)
* Os valores $z_i^{(0)} = x_i$ são as entradas
* O índice $w_{ji}$ segue a convenção "unidade destino primeiro, unidade origem depois".

Dado um conjunto de observações

$$
\mathcal{D} = \{(\mathbf{x}_n, t_n)\}_{n=1}^N
\quad
\text{onde}
\quad \mathbf{x}_n \in \mathbb{R}^{D}
$$

definimos a função de erro como a soma sobre os exemplos:

$$
E(\mathbf{w}) = \sum_{n=1}^{N} E_n(\mathbf{w})
$$

para regressão:

$$
E_n(\mathbf{w}) = \dfrac{1}{2} \sum_k (y_k(\mathbf{x}_n, \mathbf{w}) - t_{nk})^{2}
$$

para classificação:

$$
E_n(\mathbf{w}) = -\sum_k t_{nk} \ln y_k(\mathbf{x}_n)
$$

Atualizamos o valor dos pesos usando SGD dado por:

$$
\mathbf{w}^{(\tau + 1)} = \mathbf{w}^{(\tau)} - \eta\,\nabla E(\mathbf{w}^{(\tau)})
$$

precisamos então determinar o valor de $\nabla_{\mathbf{w}}\,E$.

O cálculo do gradiente requer calcular a derivada de $E_n$ em relação a cada peso $w_{ji}^{(\ell)}$.

Usando a regra da cadeia para derivadas parciais, temos que:

$$
\dfrac{\partial E_n}{\partial w_{ji}^{(\ell)}} = \dfrac{\partial E_n}{\partial a_j^{(\ell)}} \cdot \dfrac{\partial a_j^{(\ell)}}{\partial w_{ji}^{(\ell)}}
$$

Definimos então a seguinte notação:

$$
\delta_j^{(\ell)} \equiv \dfrac{\partial E_n}{\partial a_j^{(\ell)}}
$$

Dado que $a_j^{(\ell)} = \sum_i w_{ji}^{(\ell)} \cdot z_{i}^{(\ell-1)}$, temos que $\dfrac{\partial a_j^{(\ell)}}{\partial w_{ji}^{(\ell)}} = z_i^{(\ell-1)}$.

Substituindo os resultados na regra da cadeia, temos:
$$
\dfrac{\partial E_n}{\partial w_{ji}^{(\ell)}} = \delta_j^{(\ell)} \cdot z_i^{(\ell-1)}
$$

o problema agora consiste em como calcular $\delta_j^{(\ell)}$ para todas as camadas.

### **Backpropagation**

O algoritmo backpropagation calcula eficientemente $\delta_j^{(\ell)}$
para todas as camadas, explorando a estrutura do grafo em duas fases:

- **Forward pass**: propaga o sinal de entrada $\mathbf{x}_n$ camada a camada, armazenando
  as pré-ativações $a_j^{(\ell)}$ e as ativações $z_j^{(\ell)}$.
- **Backward pass**: propaga o sinal de erro da saída em direção à entrada,
  calculando os $\delta_j^{(\ell)}$ por recorrência.

Segue então o passo a passo do backpropagation:

**Passo 1 — Forward pass**

Propagar $\mathbf{x}_n$ pela rede, calculando e **armazenando** para cada camada $\ell = 1, \ldots, L$:

$$
a_j^{(\ell)} = \sum_i w_{ji}^{(\ell)} z_i^{(\ell-1)}
\quad
\text{ e }
\quad
z_j^{(\ell)} = h\left(a_j^{(\ell)}\right)
$$

Os valores $z_j^{(\ell)}$ precisam ser retidos em memória pois são usados no Passo 4.

**Passo 2 — Deltas na camada de saída**

Para a camada de saída $L$, o delta é obtido do seguinte modo:

$$
\delta_k^{(L)} = y_k - t_k
$$

**Passo 3 — Retropropagação dos deltas**

Para as camadas $\ell = (L-1), \ldots, 1$, calculamos os deltas usando a equação fundamental:

$$
\delta_j^{(\ell)} = h'\!\left(a_j^{(\ell)}\right) \sum_k w_{kj}^{(\ell+1)}\, \delta_k^{(\ell+1)}
$$


**Passo 4 — Cálculo dos gradientes**

Com os deltas calculados, o gradiente em relação a cada peso é:

$$
\frac{\partial E_n}{\partial w_{ji}^{(\ell)}} = \delta_j^{(\ell)} \cdot z_i^{(\ell-1)}
$$

**Passo 5 — Atualização dos pesos**

Acumulando os gradientes sobre **todos os exemplos** antes de atualizar (*batch gradient descent*):

$$
w_{ji}^{(\ell)} \;\leftarrow\;  w_{ji}^{(\ell)} - \eta \sum_n \delta_{j,n}^{(\ell)}\, z_{i,n}^{(\ell-1)}
$$

Ou atualizando imediatamente após **cada exemplo** (*SGD puro*):

$$
w_{ji}^{(\ell)} \;\leftarrow\;  w_{ji}^{(\ell)} - \eta\; \delta_{j,n}^{(\ell)}\, z_{i,n}^{(\ell-1)}
$$

A implementação abaixo usa SGD puro.

> BISHOP, Christopher M. Pattern Recognition and Machine Learning. New York: Springer, 2006. Seção 5.3, p. 241.

> FACELI, Katti; LORENA, Ana Carolina; GAMA, João; CARVALHO, André C. P. L. F. de. Inteligência Artificial: Uma Abordagem de Aprendizado de Máquina. Rio de Janeiro: LTC, 2011. Seções 7.1.4 e 7.1.5, p. 115 e 117.